# 02 - Data Cleaning
## Airbnb Price Prediction - Austin, TX

**Purpose of this notebook**  
Transform the raw `listings.csv` into a clean, focused dataset ready for feature 
engineering. Every decision made here is grounded in observations from `01_EDA.ipynb` 
and the inference-time constraint defined in the project specification.

**What this notebook does NOT do**  
- No feature engineering (deferred to `03_Features.ipynb`)  
- No encoding, scaling, or train/test split (deferred to `04_Preprocessing.ipynb`)  
- No `log1p` transformation of the target (deferred to `04_Preprocessing.ipynb`)

**Inference-time constraint**  
> This project predicts prices for new listings with no history. Any feature 
> unavailable at inference time is excluded from training to avoid train-serving skew.

**Input**: `data/raw/Austin/listings.csv.gz` (10,533 rows × 79 columns)  
**Output**: `data/processed/Austin/data_cleaned.parquet`

**Outline**
1. Setup & Data Loading
2. Immediate Drops - Pure Metadata
3. Feature Selection - The Inference Test
4. Cleaning - Price
5. Cleaning - Bathrooms
6. Cleaning - Missing Values
7. Final Audit & Export

---
## 1. Setup & Data Loading

We reload the raw `listings.csv` exactly as it was read in Notebook 01, with no 
prior transformation. All cleaning operations in this notebook start from the 
original data to keep the pipeline reproducible end-to-end.²

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

SEED = 42
np.random.seed(SEED)

print('Libraries loaded')

Libraries loaded


In [2]:
LISTINGS_PATH = '../data/raw/Austin/listings.csv.gz'

df = pd.read_csv(LISTINGS_PATH, low_memory=False)

print(f'Loaded listings.csv {df.shape[0]:,} rows × {df.shape[1]} columns')

Loaded listings.csv 10,533 rows × 79 columns


---
## 2. Immediate Drops - Pure Metadata

Before applying the inference test, we remove columns that have no predictive 
value by construction. These fall into three groups:

**Scraping metadata**: identifiers and timestamps generated by the Inside Airbnb 
scraper itself (`id`, `scrape_id`, `last_scraped`, `source`). They describe the 
data collection process, not the listing.

**URLs and raw text**: links and free-text fields (`listing_url`, `picture_url`, 
`name`, `description`, `neighborhood_overview`). Text features could theoretically 
carry signal via NLP, but that is out of scope for this project. The host's 
public bio and listing description are also excluded for the same reason.

**Redundant location text**: human-readable location fields (`neighbourhood`, 
`neighbourhood_group_cleansed`) that duplicate or are less reliable than 
`neighbourhood_cleansed`, which uses Inside Airbnb's standardized boundaries.

Removing these now keeps subsequent sections focused on columns that could 
plausibly carry signal.

In [3]:
print('All columns in the raw dataset:')
for col in df.columns:
    print(f'  - {col}')

All columns in the raw dataset:
  - id
  - listing_url
  - scrape_id
  - last_scraped
  - source
  - name
  - description
  - neighborhood_overview
  - picture_url
  - host_id
  - host_url
  - host_name
  - host_since
  - host_location
  - host_about
  - host_response_time
  - host_response_rate
  - host_acceptance_rate
  - host_is_superhost
  - host_thumbnail_url
  - host_picture_url
  - host_neighbourhood
  - host_listings_count
  - host_total_listings_count
  - host_verifications
  - host_has_profile_pic
  - host_identity_verified
  - neighbourhood
  - neighbourhood_cleansed
  - neighbourhood_group_cleansed
  - latitude
  - longitude
  - property_type
  - room_type
  - accommodates
  - bathrooms
  - bathrooms_text
  - bedrooms
  - beds
  - amenities
  - price
  - minimum_nights
  - maximum_nights
  - minimum_minimum_nights
  - maximum_minimum_nights
  - minimum_maximum_nights
  - maximum_maximum_nights
  - minimum_nights_avg_ntm
  - maximum_nights_avg_ntm
  - calendar_updated
  - ha

In [4]:
immediate_drops = [
    # Scraping metadata
    'id', 'scrape_id', 'last_scraped', 'source',
    'calendar_last_scraped', 'calendar_updated',

    # URLs (listing + host)
    'listing_url', 'picture_url',
    'host_url', 'host_thumbnail_url', 'host_picture_url',

    # Raw text - NLP out of scope
    'name', 'description', 'neighborhood_overview',
    'host_name', 'host_about',

    # Host identifier (pure scraping ID)
    'host_id',

    # Redundant location text
    'neighbourhood', 'neighbourhood_group_cleansed',
]

# Safety check before dropping (catches snapshot-to-snapshot renames)
missing = [c for c in immediate_drops if c not in df.columns]
assert not missing, f'Columns not found in df: {missing}'

df = df.drop(columns=immediate_drops)
print(f'Dropped {len(immediate_drops)} metadata columns')
print(f'Remaining: {df.shape[0]:,} rows × {df.shape[1]} columns')

Dropped 19 metadata columns
Remaining: 10,533 rows × 60 columns


---
## 3. Feature Selection - The Inference Test

This section is the single point of reference for column selection in the entire 
pipeline. Every column not in the final feature set is dropped here, with a short 
justification grouped by reason.

**The inference test**  
For each remaining column, we ask: *"Can a host who has never listed their property 
answer this question?"* If the answer is no, the column is dropped to avoid 
train-serving skew. A model that learns to depend on review scores will fail 
silently at inference time when those columns are zero or missing.

**Columns retained** (14 features, per the project specification)

| Type | Columns |
|---|---|
| Numeric (raw) | `accommodates`, `bedrooms`, `beds`, `availability_365` |
| Numeric (needs cleaning) | `price`, `bathrooms_text` |
| Categorical | `room_type`, `property_type`, `amenities` |
| Location | `latitude`, `longitude`, `neighbourhood_cleansed` |

`amenities`, `latitude`, and `longitude` are kept here as raw inputs. Notebook 03 
will transform them into the engineered features `amenity_tier`, `geo_cluster`, 
`distance_to_downtown_km`, and `listings_density_500m`.

**Columns dropped** are grouped by reason in the cells below.

**Drop group 1 - Review-based features**  
All review columns (`number_of_reviews*`, `review_scores_*`, `first_review`, 
`last_review`, `reviews_per_month`) are zero or NaN for a brand-new listing.

**Drop group 2 - Host history features**  
`host_since`, `host_is_superhost`, `host_response_time`, `host_response_rate`, 
`host_acceptance_rate`, `host_listings_count`, `host_total_listings_count`, 
`host_verifications`, `host_has_profile_pic`, `host_identity_verified`, 
`host_location`, `host_neighbourhood` all require a track record a new host 
does not have.

**Drop group 3 - Scraper-derived aggregates**  
`calculated_host_listings_count*`, `estimated_occupancy_l365d`, 
`estimated_revenue_l365d`, `number_of_reviews_ly`, `availability_eoy` are computed 
by Inside Airbnb from the full marketplace state at scrape time. They are not 
something a host configures.

**Drop group 4 - Redundant availability windows**  
`availability_30`, `availability_60`, `availability_90` are sub-windows of 
`availability_365`. Keeping only the yearly figure avoids multicollinearity.

**Drop group 5 - Redundant min/max nights derivatives**  
`minimum_minimum_nights`, `maximum_minimum_nights`, `minimum_maximum_nights`, 
`maximum_maximum_nights`, `minimum_nights_avg_ntm`, `maximum_nights_avg_ntm` are 
aggregates derived from `minimum_nights` and `maximum_nights`. Neither the raw 
nor the derivatives are in the final feature set, so all of them are dropped.

**Drop group 6 - Booking configuration not in final feature set**  
`minimum_nights`, `maximum_nights`, `instant_bookable`, `has_availability`, 
`license` are listing settings a host could in principle answer, but they are not 
part of the 14 retained features defined in the project specification. They are 
dropped here for consistency.

**Numeric `bathrooms` column**  
The raw `bathrooms` column is entirely empty in recent Inside Airbnb snapshots; 
`bathrooms_text` carries the actual information and is parsed in Section 5. The 
empty column is dropped here.

**Bathrooms - keep numeric, drop text**  
The raw `bathrooms` column is already a clean numeric parse of `bathrooms_text` 
(10,527 vs 10,501 non-null in this snapshot, with no rows where only the text is 
filled). The text column does carry a qualitative signal (`shared` vs `private`), 
but the project specification retains only a numeric `bathrooms_clean` feature, 
so `bathrooms_text` is dropped here for consistency with the final feature set.

In [5]:
print(f"'bathrooms' non-null count: {df['bathrooms'].notna().sum()}")
print(f"'bathrooms_text' non-null count: {df['bathrooms_text'].notna().sum()}")

'bathrooms' non-null count: 10527
'bathrooms_text' non-null count: 10501


In [6]:
feature_selection_drops = [
    # Group 1 - Review-based (no history for a new listing)
    'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
    'first_review', 'last_review', 'reviews_per_month',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value',

    # Group 2 - Host history (a new host has none)
    'host_since', 'host_location', 'host_response_time',
    'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
    'host_neighbourhood', 'host_listings_count', 'host_total_listings_count',
    'host_verifications', 'host_has_profile_pic', 'host_identity_verified',

    # Group 3 - Scraper-derived aggregates (computed from marketplace state)
    'calculated_host_listings_count',
    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms',
    'estimated_occupancy_l365d', 'estimated_revenue_l365d',
    'number_of_reviews_ly', 'availability_eoy',

    # Group 4 - Redundant availability windows (sub-windows of availability_365)
    'availability_30', 'availability_60', 'availability_90',

    # Group 5 - Redundant min/max nights derivatives
    'minimum_minimum_nights', 'maximum_minimum_nights',
    'minimum_maximum_nights', 'maximum_maximum_nights',
    'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',

    # Group 6 - Booking config not in final feature set
    'minimum_nights', 'maximum_nights',
    'instant_bookable', 'has_availability', 'license',

    # Bathrooms - keep numeric, drop text (see markdown above)
    'bathrooms_text',
]

# Safety check
missing = [c for c in feature_selection_drops if c not in df.columns]
assert not missing, f'Columns not found in df: {missing}'

df = df.drop(columns=feature_selection_drops)
print(f'Dropped {len(feature_selection_drops)} columns by inference test')
print(f'Remaining: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nFinal columns: {list(df.columns)}')

Dropped 48 columns by inference test
Remaining: 10,533 rows × 12 columns

Final columns: ['neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'amenities', 'price', 'availability_365']


---
## State of the data before cleaning

We now have a focused dataset of 12 columns. Before applying cleaning operations, 
we audit missing values and dtypes so we know exactly what each subsequent section 
needs to address.

In [7]:
audit = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notna().sum(),
    'nan_count': df.isna().sum(),
    'nan_pct': (df.isna().mean() * 100).round(2),
})
audit = audit.sort_values('nan_count', ascending=False)
print(audit)

                          dtype  non_null  nan_count  nan_pct
price                       str     10517         16     0.15
bedrooms                float64     10520         13     0.12
beds                    float64     10524          9     0.09
bathrooms               float64     10527          6     0.06
property_type               str     10533          0     0.00
longitude               float64     10533          0     0.00
latitude                float64     10533          0     0.00
neighbourhood_cleansed    int64     10533          0     0.00
room_type                   str     10533          0     0.00
accommodates              int64     10533          0     0.00
amenities                   str     10533          0     0.00
availability_365          int64     10533          0     0.00


---
## 4. Cleaning - Price

The target variable arrives as a string (`"$1,234.00"`) with 16 missing values 
(0.15%). The cleaning has three steps:

**Step 1 - Conversion to float**  
Strip `$` and thousands separators, then cast to float64.

**Step 2 - Drop rows with missing target**  
The target cannot be imputed without leaking information. The 16 rows with no 
price are dropped. At 0.15% of the dataset this has no material impact.

**Step 3 - Winsorization at $10 / $2,500**  
The EDA showed extreme outliers in both tails (test listings at $0-9 and luxury 
properties above $2,500). Rather than drop these rows we cap them, which 
preserves the listing in the dataset while limiting its influence on the model. 
The thresholds were chosen in the EDA on the basis of the price distribution: 
values below $10 are not credible nightly rates, and the right tail thins out 
sharply beyond $2,500.

The `log1p` transformation is not applied here. It belongs to the preprocessing 
pipeline (Notebook 04) so it can be fit on the training set only.

In [8]:
# Convert price string to float
df['price'] = (
    df['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype('float64')
)

print(f"Price dtype: {df['price'].dtype}")
print(f"Price NaN count: {df['price'].isna().sum()}")
print(f"\nPrice summary:")
print(df['price'].describe())

Price dtype: float64
Price NaN count: 16

Price summary:
count   10517.00
mean      414.54
std      2896.81
min         8.00
25%        86.00
50%       135.00
75%       226.00
max     50000.00
Name: price, dtype: float64


In [9]:
rows_before = len(df)
df = df.dropna(subset=['price']).reset_index(drop=True)
rows_after = len(df)

print(f'Dropped {rows_before - rows_after} rows with missing price')
print(f'Remaining: {rows_after:,} rows')

Dropped 16 rows with missing price
Remaining: 10,517 rows


In [10]:
PRICE_FLOOR = 10
PRICE_CAP = 2500

below_floor = (df['price'] < PRICE_FLOOR).sum()
above_cap = (df['price'] > PRICE_CAP).sum()
total_affected = below_floor + above_cap
pct_affected = total_affected / len(df) * 100

print(f"Below ${PRICE_FLOOR}: {below_floor} listings ({below_floor/len(df)*100:.2f}%)")
print(f"Above ${PRICE_CAP}: {above_cap} listings ({above_cap/len(df)*100:.2f}%)")
print(f"Total affected: {total_affected} listings ({pct_affected:.2f}%)")

Below $10: 1 listings (0.01%)
Above $2500: 101 listings (0.96%)
Total affected: 102 listings (0.97%)


In [11]:
# Winsorize: cap values at the floor and ceiling rather than dropping them
# Floor $10, cap $2500 - chosen in EDA from the price distribution
# Affects 102 listings (0.97% of the dataset): 1 below floor, 101 above cap
df['price'] = df['price'].clip(lower=PRICE_FLOOR, upper=PRICE_CAP)

print(f"Price summary after winsorization:")
print(df['price'].describe())
print(f"\nNew min: ${df['price'].min():.2f}")
print(f"New max: ${df['price'].max():.2f}")

Price summary after winsorization:
count   10517.00
mean      228.13
std       326.10
min        10.00
25%        86.00
50%       135.00
75%       226.00
max      2500.00
Name: price, dtype: float64

New min: $10.00
New max: $2500.00


---
## 5. Cleaning - Bathrooms

After the Feature Selection section, only the numeric `bathrooms` column remains 
(the qualitative `bathrooms_text` was dropped). The column is already a clean 
float parse provided by Inside Airbnb, so no string parsing is needed.

Six rows have missing values (0.06%). Rather than drop them, we impute with the 
median conditional on `bedrooms`: a 2-bedroom listing typically has a different 
bathroom count than a studio. If `bedrooms` itself is missing for any of these 
rows (it may be, since `bedrooms` also has NaN), we fall back to the global 
median.

The column is renamed `bathrooms_clean` to match the project specification and 
to mark explicitly that it has been through the cleaning step.

In [12]:
missing_bath = df[df['bathrooms'].isna()]
print(f"Rows with missing bathrooms: {len(missing_bath)}")
print("\nThese rows:")
print(missing_bath[['property_type', 'room_type', 'accommodates', 'bedrooms', 'beds']])

Rows with missing bathrooms: 6

These rows:
               property_type        room_type  accommodates  bedrooms  beds
65      Private room in home     Private room             2      1.00  2.00
79               Entire home  Entire home/apt             7      3.00  7.00
118    Private room in cabin     Private room             2      1.00  1.00
126              Entire home  Entire home/apt             6      3.00  4.00
9990    Private room in home     Private room             2      1.00  1.00
10139   Private room in home     Private room             2      1.00  1.00


In [13]:
# Compute the median bathrooms grouped by bedrooms
median_by_bedrooms = df.groupby('bedrooms')['bathrooms'].transform('median')

# Fallback: global median, in case any future row has bedrooms also missing
global_median = df['bathrooms'].median()

# Impute: prefer group median, fall back to global median
df['bathrooms'] = df['bathrooms'].fillna(median_by_bedrooms).fillna(global_median)

# Rename to match the project specification
df = df.rename(columns={'bathrooms': 'bathrooms_clean'})

print(f"NaN remaining in bathrooms_clean: {df['bathrooms_clean'].isna().sum()}")
print(f"\nImputed values for the 6 affected rows:")
print(df.loc[[65, 79, 118, 126, 9990, 10139], ['bedrooms', 'bathrooms_clean']])

NaN remaining in bathrooms_clean: 0

Imputed values for the 6 affected rows:
       bedrooms  bathrooms_clean
65         1.00             1.00
79         3.00             2.00
118        1.00             1.00
126        3.00             2.00
9990       1.00             1.00
10139      1.00             1.00


---
## 6. Cleaning - Missing Values

After cleaning the price and the bathrooms, two columns still have residual 
missing values:

- `bedrooms`: 13 missing (0.12%)
- `beds`: 9 missing (0.09%)

These three columns (`bedrooms`, `beds`, `accommodates`) are tightly correlated, 
as the EDA confirmed. We exploit that correlation to impute conditionally on 
`accommodates`, which is always populated:

- `bedrooms` is imputed with the median `bedrooms` for the same `accommodates` value.
- `beds` is imputed with the median `beds` for the same `accommodates` value.

In both cases we fall back to the global median if the group median is itself 
undefined (which would only happen for an isolated `accommodates` value with all 
its members missing, which is not the case here but is good defensive coding).

In [14]:
print('=== Rows with missing bedrooms ===')
print(df[df['bedrooms'].isna()][['accommodates', 'bedrooms', 'beds', 'property_type']])

print('\n=== Rows with missing beds ===')
print(df[df['beds'].isna()][['accommodates', 'bedrooms', 'beds', 'property_type']])

=== Rows with missing bedrooms ===
      accommodates  bedrooms  beds                property_type
2574             1       NaN  1.00  Private room in rental unit
7648             4       NaN  2.00                  Entire loft
7755             6       NaN  3.00                  Entire loft
7776             2       NaN  1.00                 Entire condo
8233             4       NaN  2.00           Entire rental unit
8293             2       NaN  1.00           Entire rental unit
8317             4       NaN  2.00           Entire rental unit
8355             4       NaN  2.00                  Entire loft
8559             2       NaN  1.00                  Entire loft
8598             2       NaN  1.00           Entire rental unit
9530             3       NaN  2.00           Entire rental unit
9532             3       NaN  2.00    Room in bed and breakfast
9545             2       NaN  1.00           Entire rental unit

=== Rows with missing beds ===
       accommodates  bedrooms  beds  

**Out-of-scope segment: Campsites**  
The dataset contains 14 Campsite listings (0.13%). The concept of `beds` does 
not apply to tent or RV camping, and the price range ($24-$86, median $24) 
confirms this is a distinct low-cost segment with no overlap with the urban 
rental market the project targets. All 3 rows with missing `beds` belong to 
this segment.

These rows are dropped before imputing `beds`, both for conceptual coherence 
and because Campsites fall outside the project's intended scope.

In [ ]:
# Impute bedrooms by accommodates median
median_bedrooms_by_acc = df.groupby('accommodates')['bedrooms'].transform('median')
global_median_bedrooms = df['bedrooms'].median()

df['bedrooms'] = df['bedrooms'].fillna(median_bedrooms_by_acc).fillna(global_median_bedrooms)

print(f"NaN remaining in bedrooms: {df['bedrooms'].isna().sum()}")
print(f"\nImputed values for the 13 affected rows:")
print(df.loc[[2574, 7648, 7755, 7776, 8233, 8293, 8317, 8355, 8559, 8598, 9530, 9532, 9545],
             ['accommodates', 'bedrooms', 'beds']])

In [15]:
# Drop Campsite rows: out of project scope (urban rentals) and 'beds' is
# conceptually undefined for tent/RV camping
rows_before = len(df)
df = df[df['property_type'] != 'Campsite'].reset_index(drop=True)
print(f"Dropped {rows_before - len(df)} Campsite rows")
print(f"Remaining: {len(df):,} rows")

# Now impute beds for the rows that still have missing values
still_missing = df[df['beds'].isna()]
print(f"\nRows still missing beds: {len(still_missing)}")
print(still_missing[['accommodates', 'bedrooms', 'beds', 'property_type']])

median_beds_by_acc = df.groupby('accommodates')['beds'].transform('median')
global_median_beds = df['beds'].median()
df['beds'] = df['beds'].fillna(median_beds_by_acc).fillna(global_median_beds)

print(f"\nNaN remaining in beds: {df['beds'].isna().sum()}")

Dropped 14 Campsite rows
Remaining: 10,503 rows

Rows still missing beds: 5
       accommodates  bedrooms  beds                property_type
2588              2      1.00   NaN         Private room in home
2662              2      1.00   NaN         Private room in home
3015              1      1.00   NaN  Private room in rental unit
9571              6      3.00   NaN                 Entire condo
10358             8      4.00   NaN                  Entire home

NaN remaining in beds: 0


---
## 7. Final Audit & Export

Before exporting, we run a final audit to confirm:

1. **No residual missing values** on any of the 12 columns.
2. **Consistent dtypes** — in particular `neighbourhood_cleansed` is converted to 
   `category` so downstream code (especially `04_Preprocessing.ipynb`) treats it 
   as categorical rather than numeric. It contains ZIP-like codes which are 
   numeric by accident, not by meaning.
3. **No price outliers beyond the winsorization bounds** ($10 to $2,500).
4. **Row count consistent** with the operations performed (started at 10,533, 
   dropped 16 missing prices + 14 Campsites = 10,503 expected).

The cleaned dataset is then saved as `data_cleaned.parquet` in the processed 
data folder. Parquet is preferred over CSV: it preserves dtypes (no string 
re-parsing in Notebook 03), is smaller on disk, and reads faster.

In [16]:
print('=== Final shape ===')
print(f'{df.shape[0]:,} rows × {df.shape[1]} columns')

print('\n=== Dtype and NaN audit ===')
audit = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notna().sum(),
    'nan_count': df.isna().sum(),
})
print(audit)

print('\n=== Price bounds check ===')
print(f"Min price: ${df['price'].min():.2f}  (expected >= $10)")
print(f"Max price: ${df['price'].max():.2f}  (expected <= $2500)")

print('\n=== Row count reconciliation ===')
print(f"Raw rows:        10,533")
print(f"Missing prices:    - 16")
print(f"Campsite rows:     - 14")
print(f"Expected:        10,503")
print(f"Actual:          {df.shape[0]:,}")
assert df.shape[0] == 10503, 'Row count mismatch!'
print('✓ Row count matches')

=== Final shape ===
10,503 rows × 12 columns

=== Dtype and NaN audit ===
                          dtype  non_null  nan_count
neighbourhood_cleansed    int64     10503          0
latitude                float64     10503          0
longitude               float64     10503          0
property_type               str     10503          0
room_type                   str     10503          0
accommodates              int64     10503          0
bathrooms_clean         float64     10503          0
bedrooms                float64     10490         13
beds                    float64     10503          0
amenities                   str     10503          0
price                   float64     10503          0
availability_365          int64     10503          0

=== Price bounds check ===
Min price: $10.00  (expected >= $10)
Max price: $2500.00  (expected <= $2500)

=== Row count reconciliation ===
Raw rows:        10,533
Missing prices:    - 16
Campsite rows:     - 14
Expected:        10,503
A

In [17]:
# Convert neighbourhood_cleansed to category (it's ZIP codes, categorical by meaning)
df['neighbourhood_cleansed'] = df['neighbourhood_cleansed'].astype('category')

# Final dtype check
print('Final dtypes:')
print(df.dtypes)

# Export
import os
OUTPUT_DIR = '../data/processed/Austin'
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = f'{OUTPUT_DIR}/data_cleaned.parquet'
df.to_parquet(OUTPUT_PATH, index=False)

print(f'\n✓ Saved to {OUTPUT_PATH}')
print(f'  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

Final dtypes:
neighbourhood_cleansed    category
latitude                   float64
longitude                  float64
property_type                  str
room_type                      str
accommodates                 int64
bathrooms_clean            float64
bedrooms                   float64
beds                       float64
amenities                      str
price                      float64
availability_365             int64
dtype: object

✓ Saved to ../data/processed/Austin/data_cleaned.parquet
  Shape: 10,503 rows × 12 columns


---
## Handoff to 03_Features

The cleaned dataset contains 10,503 rows and 12 columns with no missing values. 
It is the starting point for `03_Features.ipynb`, which will:

- Engineer the 6 derived features from the project specification: 
  `distance_to_downtown_km`, `geo_cluster`, `listings_density_500m`, 
  `guests_per_bedroom`, `bathrooms_per_guest`, `amenity_tier`.
- Drop the raw source columns once their engineered counterparts are produced 
  (`latitude` / `longitude` → geographic features, `amenities` → `amenity_tier`).

The output of Notebook 03 will be `data_features.parquet`, containing the 14 
final features defined in the project specification, ready for preprocessing 
in Notebook 04.